In [1]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset
dataset = load_dataset("DeepPavlov/snips")
print(dataset)

/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['utterance', 'label'],
        num_rows: 13084
    })
    test: Dataset({
        features: ['utterance', 'label'],
        num_rows: 1400
    })
})


In [4]:
import pandas as pd

In [5]:
df = pd.DataFrame(dataset['train'])
print(df['label'].unique())
print(df['label'].value_counts())

[0 1 2 3 4 5 6]
label
2    1900
3    1900
1    1873
6    1859
4    1856
5    1854
0    1842
Name: count, dtype: int64


In [6]:
print(dataset['train'].features)
print()
# See a few examples
for i in range(5):
    print(dataset['train'][i])

{'utterance': Value('string'), 'label': Value('int64')}

{'utterance': 'Add Don and Sherri to my Meditate to Sounds of Nature playlist', 'label': 0}
{'utterance': 'put United Abominations onto my rare groove playlist', 'label': 0}
{'utterance': 'add the tune by misato watanabe to the Trapeo playlist', 'label': 0}
{'utterance': 'add this artist to my this is miguel bosé playlist', 'label': 0}
{'utterance': 'add heresy and the hotel choir to the evening acoustic playlist', 'label': 0}


In [7]:
label_map = {
    0: "AddToPlaylist",
    1: "BookRestaurant", 
    2: "GetWeather",
    3: "PlayMusic",
    4: "RateBook",
    5: "SearchCreativeWork",
    6: "SearchScreeningEvent"
}

# Verify distribution
for k, v in label_map.items():
    count = len(df[df['label'] == k])
    print(f"{v}: {count}")

AddToPlaylist: 1842
BookRestaurant: 1873
GetWeather: 1900
PlayMusic: 1900
RateBook: 1856
SearchCreativeWork: 1854
SearchScreeningEvent: 1859


In [8]:
from collections import Counter

In [9]:
# Tokenize all utterances
def tokenize(sentence):
    return sentence.lower().split()
    
# Build vocabulary
counter = Counter()
for sample in dataset['train']:
    counter.update(tokenize(sample['utterance']))

# Create word to index mapping
# 0 = padding, 1 = unknown
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.items():
    if freq >= 2:  # ignore words appearing only once
        word2idx[word] = len(word2idx)

print(f"Vocabulary size: {len(word2idx)}")

Vocabulary size: 4475


# Dataset Splitting

In [10]:
from sklearn.model_selection import train_test_split

The dataset already comes pre-split into train and test from HuggingFace.  
However, we do need a validation set, therefore, we'll split dataset['train'] to get validation set(only 10%).

In [11]:
train_data, val_data = train_test_split(
    dataset['train'].to_pandas(), 
    test_size=0.1,      # 10% for validation
    random_state=100,
    stratify=dataset['train']['label']  # keeps class balance
)
print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(dataset['test'])}")

Train: 11775
Val: 1309
Test: 1400


# Dataset Encoder

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

In [13]:
MAX_LEN = 50

def encode(sentence):
    tokens = tokenize(sentence)
    ids = [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
    # pad or truncate to MAX_LEN
    ids = ids[:MAX_LEN] + [0] * (MAX_LEN - len(ids))
    return ids

class SNIPSDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx] if hasattr(self.data, 'iloc') else self.data[idx]
        x = torch.tensor(encode(row['utterance']), dtype=torch.long)
        y = torch.tensor(row['label'], dtype=torch.long)
        return x, y

train_dataset = SNIPSDataset(train_data)
val_dataset = SNIPSDataset(val_data)
test_dataset = SNIPSDataset(dataset['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 368
Val batches: 41
Test batches: 44


In [14]:
pip install pytorch-crf

Note: you may need to restart the kernel to use updated packages.


# Model

In [15]:
import torch
from TorchCRF import CRF

In [17]:
import torch.nn as nn
from TorchCRF import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(BiLSTM_CRF, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # BiLSTM layer
        self.bilstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, num_classes)
        
        # CRF layer
        self.crf = CRF(num_classes)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)           # (batch_size, seq_len, embedding_dim)
        lstm_out, _ = self.bilstm(embedded)    # (batch_size, seq_len, hidden_dim)
        emissions = self.fc(lstm_out)          # (batch_size, seq_len, num_classes)
        return emissions

# Initialize model
VOCAB_SIZE = len(word2idx)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 7

model = BiLSTM_CRF(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model)

BiLSTM_CRF(
  (embedding): Embedding(4475, 128, padding_idx=0)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=7, bias=True)
  (crf): CRF(num_tags=7)
)


# Optimizer

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for x, y in loader:
        optimizer.zero_grad()
        emissions = model(x)                  # (batch, seq_len, num_classes)
        loss = -model.crf(emissions, y.unsqueeze(1).expand_as(emissions[:,:,0]))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

print("Training setup ready")

Training setup ready


In [26]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            emissions = model(x)
            predictions = model.crf.decode(emissions)
            # take first token prediction as intent label
            preds = torch.tensor(predictions[0])
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [27]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/10 | Loss: 340.1517 | Val Acc: 0.9756
Epoch 2/10 | Loss: 206.3071 | Val Acc: 0.9740
Epoch 3/10 | Loss: 180.3605 | Val Acc: 0.9778
Epoch 4/10 | Loss: 157.4036 | Val Acc: 0.9763
Epoch 5/10 | Loss: 96.6324 | Val Acc: 0.9725
Epoch 6/10 | Loss: 116.0329 | Val Acc: 0.9748
Epoch 7/10 | Loss: 111.1092 | Val Acc: 0.9733
Epoch 8/10 | Loss: 92.3061 | Val Acc: 0.9725
Epoch 9/10 | Loss: 76.7160 | Val Acc: 0.9717
Epoch 10/10 | Loss: 117.7430 | Val Acc: 0.9748


In [28]:
test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.9764
